# Wall Surface Defect Detection

This notebook demonstrates **wall surface defect detection and texture change analysis**.

### What it detects
| Defect Type | Description |
|---|---|
| Hairline Cracks | Fine network of cracks |
| Stain / Contamination | Embedded stains or foreign material |
| Paint Patch / Discoloration | Irregular paint patches or colour changes |
| Surface Pitting | Small pits or erosion |
| Peeling / Scaling | Paint peel or surface scaling |
| Writing / Scratch Marks | Graffiti, gouges, surface markings |

### Detection modes available
- **`texture`** — Lightweight, no model required (SSIM + contour analysis)
- **`yolo`** — Fine-tuned YOLOv8 on [NEU Surface Defect dataset](https://github.com/abin24/Surface-Inspection-defect-detection-dataset)
- **`patchcore`** — Unsupervised anomaly detection via [anomalib](https://github.com/openvinotoolkit/anomalib)

Source: [`wall_defect_detector.py`](../wall_defect_detector.py)

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from IPython.display import display

from wall_defect_detector import WallDefectDetector, NEU_CLASSES, DEFECT_DISPLAY_NAMES

print('All imports OK')

## Configuration

In [ ]:
WALL_BEFORE = '../Image_cc.jpg'   # Before image (or a wall photo)
WALL_AFTER  = '../Image2_cc.png'  # After image
OUTPUT_DIR  = '../outputs'

# Detection mode: 'texture' (no model) | 'yolo' (fine-tuned) | 'patchcore'
DETECTION_MODE = 'texture'

# If using 'yolo' mode, set the path to your fine-tuned model:
DEFECT_MODEL_PATH = None  # e.g. '../runs/wall_defect_v1/weights/best.pt'

import os; os.makedirs(OUTPUT_DIR, exist_ok=True)

## About the Surface Inspection Dataset

The [Surface Inspection Defect Detection Dataset](https://github.com/abin24/Surface-Inspection-defect-detection-dataset) contains multiple industrial surface categories.

For wall surface detection we use the **NEU Steel Surface Defect** subset which maps naturally to wall defect types:

In [ ]:
mapping = [
    ('crazing',         'Hairline Cracks',           'Fine crack network on wall plaster/paint'),
    ('inclusion',       'Stain / Contamination',     'Foreign material embedded in surface'),
    ('patches',         'Paint Patch / Discoloration','Irregular paint/coating patches'),
    ('pitted_surface',  'Surface Pitting',           'Erosion, small holes in wall surface'),
    ('rolled_in_scale', 'Peeling / Scaling',         'Paint peeling or surface layer scaling'),
    ('scratches',       'Writing / Scratch Marks',   'Graffiti, gouges, surface markings'),
]

df_map = pd.DataFrame(mapping, columns=['NEU Dataset Class', 'Wall Defect Type', 'Description'])
display(df_map.style.set_caption('NEU Steel → Wall Surface Defect Mapping')
        .set_table_styles([{'selector':'caption','props':[('font-size','14px'),('font-weight','bold')]}])
        .hide(axis='index'))

## Model Options: Which model should you use?

| Mode | Model | Training Needed | Best For |
|---|---|---|---|
| `texture` | None (SSIM + contours) | No | Quick demos, no GPU |
| `yolo` | Fine-tuned YOLOv8 | Yes – on NEU dataset | Labelled defect types |
| `patchcore` | anomalib PatchCore | Yes – unsupervised | Unknown defect types |

### Recommendation
- Start with **`texture`** mode for a quick demo
- Use **`yolo`** after fine-tuning on the NEU dataset for production
- Use **`patchcore`** when you don't know what defects to expect

## 1. Initialise Detector

In [ ]:
detector = WallDefectDetector(
    mode=DETECTION_MODE,
    model_path=DEFECT_MODEL_PATH,
    confidence_threshold=0.30
)

## 2. Load Wall Images

In [ ]:
img_before = cv2.imread(WALL_BEFORE)
img_after  = cv2.imread(WALL_AFTER)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(img_before, cv2.COLOR_BGR2RGB))
axes[0].set_title('BEFORE', fontsize=14, fontweight='bold'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(img_after, cv2.COLOR_BGR2RGB))
axes[1].set_title('AFTER', fontsize=14, fontweight='bold');  axes[1].axis('off')
plt.suptitle('Wall Images', fontsize=16)
plt.tight_layout(); plt.show()

## 3. Detect Defects in Each Frame

In [ ]:
defects_before = detector.detect_defects(img_before)
defects_after  = detector.detect_defects(img_after)

print(f'Defects in BEFORE image: {len(defects_before)}')
for d in defects_before:
    print(f"  {d['display_name']:30s}  conf={d['confidence']:.2f}  area={d['area']}px²")

print(f'\nDefects in AFTER image: {len(defects_after)}')
for d in defects_after:
    print(f"  {d['display_name']:30s}  conf={d['confidence']:.2f}  area={d['area']}px²")

## 4. Visualise Defects

In [ ]:
vis_before = detector.visualize_defects(img_before, defects_before, title='BEFORE')
vis_after  = detector.visualize_defects(img_after,  defects_after,  title='AFTER')

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].imshow(cv2.cvtColor(vis_before, cv2.COLOR_BGR2RGB))
axes[0].set_title(f'Before – {len(defects_before)} defect(s)', fontsize=13); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(vis_after, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'After – {len(defects_after)} defect(s)',  fontsize=13); axes[1].axis('off')
plt.suptitle('Wall Surface Defect Detection', fontsize=16)
plt.tight_layout(); plt.show()

## 5. Texture Change Analysis (SSIM + Colour)

In [ ]:
texture = detector.compute_texture_diff(img_before, img_after)

print('Texture Analysis Results')
print('='*40)
print(f"  SSIM Score        : {texture['ssim_score']}  (1.0 = identical)")
print(f"  Colour Similarity : {texture['color_similarity']}  (1.0 = same colour)")
print(f"  Changed Area      : {texture['change_percentage']}%")
print(f"  Texture Changed?  : {'YES' if texture['texture_changed'] else 'NO'}")

# Visualise the change mask
mask_colour = cv2.applyColorMap(texture['change_mask'], cv2.COLORMAP_HOT)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(cv2.cvtColor(img_before, cv2.COLOR_BGR2RGB)); axes[0].set_title('Before'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(img_after,  cv2.COLOR_BGR2RGB)); axes[1].set_title('After');  axes[1].axis('off')
axes[2].imshow(cv2.cvtColor(mask_colour, cv2.COLOR_BGR2RGB))
axes[2].set_title(f"Texture Change Mask\n(changed area: {texture['change_percentage']}%)"); axes[2].axis('off')
plt.suptitle('Texture / Paint Change Analysis', fontsize=15)
plt.tight_layout(); plt.show()

## 6. Before vs After Defect Comparison

In [ ]:
comparison = detector.compare_wall_surfaces(img_before, img_after)

print('='*60)
print('WALL SURFACE CHANGE SUMMARY')
print('='*60)
print(comparison['summary'])
print(f"\nNew defects       : {len(comparison['new_defects'])}")
print(f"Resolved defects  : {len(comparison['resolved_defects'])}")
print(f"Persisting defects: {len(comparison['persisting_defects'])}")

## 7. Side-by-side Comparison Visualisation

In [ ]:
comparison_vis = detector.visualize_comparison(img_before, img_after, comparison)

plt.figure(figsize=(20, 8))
plt.imshow(cv2.cvtColor(comparison_vis, cv2.COLOR_BGR2RGB))
plt.axis('off')

legend = [
    mpatches.Patch(color='lime',   label='NEW defect'),
    mpatches.Patch(color='red',    label='RESOLVED defect'),
    mpatches.Patch(color='yellow', label='EXISTING (persisting)'),
]
plt.legend(handles=legend, loc='lower center', ncol=3, fontsize=12,
           bbox_to_anchor=(0.5, -0.06))
plt.title('Wall Surface: Before vs After Comparison', fontsize=16)
plt.tight_layout(); plt.show()

cv2.imwrite(f'{OUTPUT_DIR}/wall_defect_comparison.jpg', comparison_vis)
print('Saved to outputs/wall_defect_comparison.jpg')

## 8. Defect Summary Table

In [ ]:
rows = []
for d in comparison['new_defects']:
    rows.append({'Status': '🔴 NEW',       'Defect Type': d['display_name'], 'Confidence': f"{d['confidence']:.2f}", 'Area (px²)': d['area']})
for d in comparison['resolved_defects']:
    rows.append({'Status': '🟢 RESOLVED',  'Defect Type': d['display_name'], 'Confidence': f"{d['confidence']:.2f}", 'Area (px²)': d['area']})
for p in comparison['persisting_defects']:
    d = p['before']
    rows.append({'Status': '🟡 PERSISTING','Defect Type': d['display_name'], 'Confidence': f"{d['confidence']:.2f}", 'Area (px²)': d['area']})

if rows:
    df = pd.DataFrame(rows)
    display(df.style.set_caption('Wall Surface Defect Report')
            .set_table_styles([{'selector':'caption','props':[('font-size','14px'),('font-weight','bold')]}])
            .hide(axis='index'))
else:
    print('No defects detected in either frame.')

## Optional: Fine-tune YOLOv8 on NEU Surface Defect Dataset

Run this section only if you have downloaded the NEU dataset and want to fine-tune.

In [ ]:
# === OPTIONAL: Fine-tuning ===
# Uncomment and set the path to your downloaded NEU dataset

# from wall_defect_detector import prepare_neu_dataset, WallDefectDetector

# NEU_ROOT    = '/path/to/NEU-DET'          # downloaded dataset root
# OUTPUT_DATA = '../datasets/wall_defect_yolo'
# MODEL_OUT   = '../runs/wall_defect_v1'

# # Step 1: Prepare dataset in YOLO format
# yaml_path = prepare_neu_dataset(NEU_ROOT, OUTPUT_DATA)

# # Step 2: Fine-tune
# yolo_detector = WallDefectDetector(mode='yolo')
# yolo_detector.fine_tune_on_dataset(yaml_path, epochs=50, output_dir=MODEL_OUT)

# # Step 3: Use fine-tuned model
# fine_tuned = WallDefectDetector(mode='yolo', model_path=f'{MODEL_OUT}/wall_defect_v1/weights/best.pt')

print("Fine-tuning section is commented out. Uncomment to train.")